In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

In [3]:
from langchain_openrouter import ChatOpenRouter
model = ChatOpenRouter(model="openai/gpt-5-nano")   # or anthropic/claude-3.5-haiku, etc.

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    context_schema=ColourContext  
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='ccb29701-61bd-40bf-ac2d-aa8f40bda985'),
              AIMessage(content='I don’t know your favourite colour yet. If you’d like, I can guess or help you figure it out.\n\n- Quick guess: blue is a very common favourite.\n- Or we can do a quick 3-question quiz to determine it:\n  1) Do you prefer cool colors (blue/green/purple) or warm colors (red/orange/yellow)?\n  2) Do you like bold saturated colors or softer pastels?\n  3) Do you choose colors based on mood or the situation?\n\nAnswer either with your preferences or say “guess for me,” and I’ll suggest a colour.', additional_kwargs={'reasoning_content': '**Clarifying favorite color**\n\nThe user asked about their favorite color, but I don\'t know it yet. I should definitely ask some clarifying questions! Maybe I can suggest that we don\'t know unless they\'ve shared it before. I could offer to guess randomly or based on t

## Accessing Context

In [7]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

In [8]:
agent = create_agent(
    # model="gpt-5-nano",
    model=model,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext
)

In [9]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext()
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='4a4a3b74-e1e0-4dfd-9cc2-8ce222fea1e0'),
              AIMessage(content='', additional_kwargs={'reasoning_content': '**Preparing color retrieval**\n\nI\'m planning to call the "get_favourite_colour" tool. It looks like the schema is set up for this, so I\'ll make sure to invoke it correctly. I need to use the "commentary" channel to ensure it\'s processed properly. After I call this tool and it returns the answer, I\'ll present the final result to the user. I’m focused on getting this right, so let\'s move forward!', 'reasoning_details': [{'summary': '**Preparing color retrieval**\n\nI\'m planning to call the "get_favourite_colour" tool. It looks like the schema is set up for this, so I\'ll make sure to invoke it correctly. I need to use the "commentary" channel to ensure it\'s processed properly. After I call this tool and it returns the answer, I\'ll present the final re

In [10]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='0f4cffc0-d78b-4b31-b5c6-a28880bcbe11'),
              AIMessage(content='', additional_kwargs={'reasoning_content': '**Fetching favorite color**\n\nI need to respond to the user\'s question about their favorite color. There are some tools available, specifically get_favourite_colour. I plan to call this function without any parameters to get the answer. Once I have it, I’ll present the result as, "Your favorite colour is X." It’s essential to remember that this is a simulated function, so I\'ll have to wait for the tool to return the value before I proceed. Let\'s go ahead and call it!', 'reasoning_details': [{'summary': '**Fetching favorite color**\n\nI need to respond to the user\'s question about their favorite color. There are some tools available, specifically get_favourite_colour. I plan to call this function without any parameters to get the answer. Once I have it, 

In [11]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my least favourite colour?")]},
    context=ColourContext(favourite_colour="green")
)

pprint(response)

{'messages': [HumanMessage(content='What is my least favourite colour?', additional_kwargs={}, response_metadata={}, id='ca52222a-c5f2-46a7-a951-cb53755e5e9a'),
              AIMessage(content='', additional_kwargs={'reasoning_content': "**Responding to colour inquiry**\n\nThe user asked about their least favorite color, so I need to use the appropriate tool, which is get_least_favourite_colour. There's no need for a parallel call; just one tool call will do. I'll call functions.get_least_favourite_colour and wait for the result. It's important to handle whatever the tool returns since it could be any type, likely a color string. Afterward, I’ll present the result clearly and concisely. Let's get started!", 'reasoning_details': [{'summary': "**Responding to colour inquiry**\n\nThe user asked about their least favorite color, so I need to use the appropriate tool, which is get_least_favourite_colour. There's no need for a parallel call; just one tool call will do. I'll call functions.ge